# Machine Learning - Exercise 3
# SMS SPAM classification

To perform the experiments on the SMSSpamCollection dataset you need to set-up your Colab such that it is able to load the desired data. To achieve this, you need to perform the following actions:

*   download the dataset available at this [link](https://drive.google.com/a/diag.uniroma1.it/file/d/17YZemn1MidhFA0-wenfVolZAwclLRUXM/view)
*   copy the dataset in a folder of your personal Drive
*   mount your Google Drive (more details will follow)
*   set the correct path for loading the dataset (more details will follow)





## Import needed libraries

In [7]:
import numpy as np
import pandas as pd
import random

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import *
from sklearn.naive_bayes import *
from sklearn.metrics import confusion_matrix, classification_report

print('Libraries imported.')

Libraries imported.


## Load data

Mount Google Drive by following the instructions given at the provided link

In [8]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


To load the file set the correct path of the dataset located in your drive. Once mounted, your drive works like a Linux system, so you can check folders etc... running commands like `ls` or `cd` preceded by `%`

In [9]:
# example path of dataset copied in My Drive folder: /content/drive/My Drive/SMSSpamCollection'
filename = '/content/drive/My Drive/Colab Notebooks/MLEx3/SMSSpamCollection'
db = pd.read_csv(filename, sep='\t', header=None, names=['label', 'text'])
print('File loaded: %d samples.' %(len(db.label)))

File loaded: 5572 samples.


Show a random sample

In [10]:
id = random.randrange(0,len(db.label))
print('[%d] %s | %s' %(id,db.label[id],db.text[id]))

[1334] ham | Oh... Icic... K lor, den meet other day...


## Choose vectorizer

Compute vectorizer terms for all messages. More info:



*   [Hashing](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.HashingVectorizer.html)
*   [Count](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
*   [Tfid](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) 



source: https://iksinc.online/2019/12/21/numeric-representation-of-text-countvectorizer-to-hashingvectorizer/

A recent article in Forbes stated that unstructured data accounts for about 90% of the data being generated daily. A large part of unstructured data consists of text in the form of emails, news reports, social media postings, phone transcripts, product reviews etc. Analyzing such data for pattern discovery requires converting text to numeric representation in the form of a vector using words as features. Such a representation is known as vector space model in information retrieval; in machine learning it is known as bag-of-words (BoW) model.

    corpus = ['The sky is blue and beautiful',
    'The king is old and the queen is beautiful',
    'Love this beautiful blue sky',
    'The beautiful queen and the old king']

This section will briefly describe the main text vectorizers from sklearn library.

The **CountVectorizer** is the simplest way of converting text to vector. It tokenizes the documents to build a vocabulary of the words present in the corpus and counts how often each word from the vocabulary is present in each and every document in the corpus. Thus, every document is represented by a vector whose size equals the vocabulary size and entries in the vector for a particular document show the count for words in that document. When the document vectors are arranged as rows, the resulting matrix is called document-term matrix; it is a convenient way of representing a small corpus.
Below, an example of CountVectorizer in which we filter out common words such as 'is' and 'the' that does not provide any useful indication about the document content.

    vectorizer = CountVectorizer(stop_words='english')
    X = vectorizer.fit_transform(corpus)
    print(vectorizer.get_feature_names())
    Doc_Term_Matrix = pd.DataFrame(X.toarray(),columns= vectorizer.get_feature_names())
    Doc_Term_Matrix

For any large corpus the resulting matrix will be a sparse matrix. Thus, internally the sparse matrix representation is used to store document vectors.

One issue with the bag of words representation is the loss of context. The BoW representation just focuses on words presence in isolation; it doesn’t use the neighboring words to build a more meaningful representation. The CountVectorizer provides a way to overcome this issue by allowing a vector representation using N-grams of words. In such a model, N successive words are used as features. Thus, in a bi-gram model, N = 2, two successive words will be used as features in the vector representations of documents. The result of such a vectorization for our small corpus example is shown below. Here the parameter ngram_range = (1,2) tells the vectorizer to use two successive words along with each single word as features for the resulting vector representation.

    vectorizer = CountVectorizer(ngram_range = (1,2),stop_words='english')
    X = vectorizer.fit_transform(corpus)
    print(vectorizer.get_feature_names())
    Doc_Term_Matrix = pd.DataFrame(X.toarray(),columns= vectorizer.get_feature_names())
    Doc_Term_Matrix

This comes at the cost of increased vector size.

Simply using the word count as a feature value of a word really doesn’t reflect the importance of that word in a document. For example, if a word is present frequently in all documents in a corpus, then its count value in different documents is not helpful in discriminating between different documents. On other hand, if a word is present only in a few of documents, then its count value in those documents can help discriminating them from the rest of the documents. Thus, the importance of a word, i.e. its feature value, for a document not only depends upon how often it is present in that document but also how is its overall presence in the corpus. This notion of importance of a word in a document is captured by a scheme, known as the **term frequency-inverse document frequency (tf-idf)** weighting scheme.

The sklearn library offers two ways to generate the tf-idf representations of documents. 

The **TfidfTransformer** transforms the count values produced by the CountVectorizer to tf-idf weights.

    from sklearn.feature_extraction.text import TfidfTransformer
    transformer = TfidfTransformer()
    tfidf = transformer.fit_transform(X)
    Doc_Term_Matrix = pd.DataFrame(tfidf.toarray(),columns= vectorizer.get_feature_names())
    pd.set_option("display.precision", 2)
    Doc_Term_Matrix

Another way is to use the **TfidfVectorizer** which combines both counting and term weighting in a single class.

    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(ngram_range = (1,2),stop_words='english')
    tfidf = vectorizer.fit_transform(corpus)
    Doc_Term_Matrix = pd.DataFrame(tfidf.toarray(),columns= vectorizer.get_feature_names())
    Doc_Term_Matrix
  
There are two main issues with the CountVectorizer and TdidfVectorizer. First, the vocabulary size can grow so much so as not to fit in the available memory for large corpus. In such a case, we need two passes over data. If we were to distribute the vectorization task to several computers, then we will need to synchronize vocabulary building across computing nodes. The other issue arises in the context of an online text classifier built using the count vectorizer, for example spam classifier which needs to decide whether an incoming email is spam or not. When such a classifier encounters words not in its vocabulary, it ignores them. A spammer can take advantage of this by deliberately misspelling words in its message which when ignored by the spam filter will cause the spam message appear normal.  The **HashingVectorizer** overcomes these limitations.

The HashingVectorizer maintains no vocabulary and determines the index of a word in an array of fixed size via hashing. Since no vocabulary is maintained, the presence of new or misspelled words doesn’t create any problem. Also the hashing is done on the fly and memory need is diminshed.

In [14]:
vectorizer_type = "count" # "hashing", "count" or "tfid"

if vectorizer_type == "hashing":
  vectorizer = HashingVectorizer(stop_words='english') # multivariate
elif vectorizer_type == "count":
  vectorizer = CountVectorizer(stop_words='english') # multinomial
elif vectorizer_type == "tfid":
  vectorizer = TfidfVectorizer(stop_words='english')

X_all = vectorizer.fit_transform(db.text)
y_all = db.label

print(X_all.shape)
print(y_all.shape)
 

(5572, 8444)
(5572,)


## Split data

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, 
          test_size=0.2, random_state=16)

print("Train: %d - Test: %d" %(X_train.shape[0],X_test.shape[0]))

#id = random.randrange(0,X_train.shape[0])
#print('%d ' %(id))
#print('%d %s %s' %(id,str(y_train[id]),str(X_train[id])))


Train: 4457 - Test: 1115


## Create and fit Model

**Multinomial logistic regression** is a classification method that generalizes logistic regression to multiclass problems, i.e. with more than two possible discrete outcomes. 



In [16]:
model_type = "multinomial" # "bernoulli" or "multinomial"

if model_type == "bernoulli":
  model = BernoulliNB().fit(X_train, y_train)
  print('Bernoulli Model created')
elif model_type == "multinomial":
  model = MultinomialNB().fit(X_train, y_train)
  print('Multinomial Model created')

Multinomial Model created


## Evaluation

In [17]:
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[962   9]
 [ 10 134]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       971
        spam       0.94      0.93      0.93       144

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## Prediction

In [18]:
smsnew1 = np.array(['Hello, did you solve ML exercise?'])
xnew1 = vectorizer.transform(smsnew1)
ynew1 = model.predict(xnew1)
print('%s %s' %(smsnew1,ynew1))

smsnew2 = np.array(['You won $1,000! Call now 1-800-1234567'])
xnew2 = vectorizer.transform(smsnew2)
ynew2 = model.predict(xnew2)
print('%s %s' %(smsnew2,ynew2))

['Hello, did you solve ML exercise?'] ['ham']
['You won $1,000! Call now 1-800-1234567'] ['spam']


## Home Exercises

**Question 1**

Design and implement an evaluation procedure to assess and compare the performance of the three vectorizers and the two models proposed above.




In [85]:
from sklearn.metrics import confusion_matrix

# is thys pythonic enough?
def better_confusion_matrix (y_true, y_pred, target_names=None):

  cm = confusion_matrix(y_true, y_pred)

  if target_names is not None and cm.shape[0] != len(target_names):
    raise Exception("Invalid array for target names.")

  formatter='{:>12}'
  st = '\n' + (formatter.format(' ') + ' '.join([formatter.format(item) for item in target_names]) + '\n' if target_names is not None else '') + '\n'
  st += '\n'.join([(formatter.format(target_names[indx]) if target_names is not None else '') + ''.join([formatter.format(item) for item in row]) for indx, row in enumerate(cm)])
  return st + '\n'


In [86]:
import numpy as np
import pandas as pd
import random

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import *
from sklearn.naive_bayes import *
from sklearn.metrics import classification_report

from google.colab import drive
drive.mount('/content/drive/')

# dataset
filename = '/content/drive/My Drive/Colab Notebooks/MLEx3/SMSSpamCollection'
db = pd.read_csv(filename, sep='\t', header=None, names=['label', 'text'])
print('File loaded: %d samples.' %(len(db.label)))

# same dataset with different representation (vectorizers)
X_all_count = CountVectorizer(stop_words='english').fit_transform(db.text)
X_all_tfid = TfidfVectorizer(stop_words='english').fit_transform(db.text)

# 8444 is the number of tokens (without english stopwords) present in the corpus
# we know this since the countvectorizer (and tfidvectorizer) uses an array with 
# a cell for each token
#
# the hashingvectorizer uses a fixed length array and we can specify its length
# with the n_features parameter. We should get no collisions

X_all_hash = HashingVectorizer(n_features=X_all_count.shape[1], stop_words='english').fit_transform(db.text)

y_all = db.label

X_train_count, X_test_count, y_train, y_test = train_test_split(X_all_count, y_all, \
                                     test_size=0.2, random_state=16)

X_train_tfid, X_test_tfid, y_train, y_test = train_test_split(X_all_tfid, y_all, \
                                     test_size=0.2, random_state=16)

X_train_hash, X_test_hash, y_train, y_test = train_test_split(X_all_hash, y_all, \
                                     test_size=0.2, random_state=16)

X_train = {}
X_train["hash"] = X_train_hash
X_train["count"] = X_train_count
X_train["tfid"] = X_train_tfid

# model
model_type = "bernoulli" # "bernoulli" or "multinomial"
vectorizer_type = "count" # "hash", "count" or "tfid"

if model_type == "bernoulli":
  model = BernoulliNB().fit(X_train[vectorizer_type], y_train)
  print('Bernoulli Model created')
elif model_type == "multinomial":
  model = MultinomialNB().fit(X_train[vectorizer_type], y_train)
  print('Multinomial Model created')

y_pred_hash = model.predict(X_test_hash)
y_pred_count = model.predict(X_test_count)
y_pred_tfid = model.predict(X_test_tfid)

target_names = ['HAM', 'SPAM']

# precision = ability to avoid false positives = TP / (TP + FP) 
#             (elements belonging to a different class classified as elements of
#             the current one)
# recall = ability to avoid false negatives = TP / (TP + FN)
#             (true positives over all the positives)

print("Hashing vectorizer:")
print(better_confusion_matrix(y_test, y_pred_hash, target_names))
print(classification_report(y_test, y_pred_hash, target_names=target_names))

print("Count vectorizer:")
print(better_confusion_matrix(y_test, y_pred_count, target_names))
print(classification_report(y_test, y_pred_count, target_names=target_names))

print("Tfid vectorizer:")
print(better_confusion_matrix(y_test, y_pred_tfid, target_names))
print(classification_report(y_test, y_pred_tfid, target_names=target_names))

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
File loaded: 5572 samples.
Bernoulli Model created
Hashing vectorizer:

                     HAM         SPAM

         HAM         970           1
        SPAM         143           1

              precision    recall  f1-score   support

         HAM       0.87      1.00      0.93       971
        SPAM       0.50      0.01      0.01       144

    accuracy                           0.87      1115
   macro avg       0.69      0.50      0.47      1115
weighted avg       0.82      0.87      0.81      1115

Count vectorizer:

                     HAM         SPAM

         HAM         967           4
        SPAM          21         123

              precision    recall  f1-score   support

         HAM       0.98      1.00      0.99       971
        SPAM       0.97      0.85      0.91       144

    accuracy                           0.98      1115
   ma